# STA baseline: data and evaluation smoke test

This notebook checks the short-term action anticipation (STA) pipeline before model training. It loads the saved participant split (or recreates the exploration notebook's deterministic split), samples 16 frames across the five-second observation window from one locally available training video, resizes them to 640 x 360, scales RGB values to [0, 1], and verifies that every sampled frame is before the required one-second gap. Augmentation is intentionally omitted for this smoke test. It then evaluates a training-frequency prior on the validation labels.

The prior needs no video input; the local clip is only a smoke test of the video/timestamp path. Run this notebook on VSC with the full data to test the same code against the complete collection. Do not use hidden-test labels.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
from pathlib import Path


for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "README.md").exists() and (candidate / "src" / "data_processing.py").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the project root.")


sys.path.insert(0, str(PROJECT_ROOT))
from src.data_processing import (
    DEFAULT_RAW_DIR,
    find_annotation_file,
    index_videos,
    load_annotations,
    match_available_annotations,
    participant_split,
    read_frame,
    timestamp_to_seconds,
)

RAW_DIR = DEFAULT_RAW_DIR
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
STA_CONTEXT_SECONDS = 5.0
STA_GAP_SECONDS = 1.0
STA_FRAME_COUNT = 16
FRAME_WIDTH = 640
FRAME_HEIGHT = 360
NUM_VERBS = 97
NUM_NOUNS = 300

print(f"Project root: {PROJECT_ROOT}")

## 1. Load annotations and the fixed participant split

Use the saved split when present. If it is absent on a fresh VSC checkout, recreate it with the same sorted-participant, 20%, seed-42 procedure as the exploration notebook.

In [ ]:
annotation_path = find_annotation_file(RAW_DIR)
annotations = load_annotations(annotation_path)
if annotations["narration_id"].duplicated().any():
    raise ValueError("Training annotations contain duplicate narration_id values.")

train_path = PROCESSED_DIR / "EPIC_100_train_participant_split.csv"
validation_path = PROCESSED_DIR / "EPIC_100_validation_participant_split.csv"
if train_path.exists() and validation_path.exists():
    train_df = pd.read_csv(train_path)
    validation_df = pd.read_csv(validation_path)
else:
    train_df, validation_df = participant_split(annotations, validation_fraction=0.20, seed=42)

train_participants = set(train_df["participant_id"])
validation_participants = set(validation_df["participant_id"])
assert train_participants.isdisjoint(validation_participants), "Participant leakage between splits."
assert set(train_df["narration_id"]).isdisjoint(validation_df["narration_id"]), "Event leakage between splits."
assert not train_df.empty and not validation_df.empty

split_summary = pd.DataFrame(
    {
        "events": [len(train_df), len(validation_df)],
        "participants": [len(train_participants), len(validation_participants)],
        "verbs": [train_df["verb_class"].nunique(), validation_df["verb_class"].nunique()],
        "nouns": [train_df["noun_class"].nunique(), validation_df["noun_class"].nunique()],
    },
    index=["train", "validation"],
)
display(split_summary)

## 2. Sample a legal STA observation

For an action starting at `t`, the observation spans `max(0, t - 6)` to `t - 1` seconds. Sample requests exclude the right endpoint, and the decoded frame timestamps are checked against the cutoff. An event with `t <= 1` has no legal observation and must use a prior-only prediction.

In [ ]:
def sta_window(action_start, context_seconds=STA_CONTEXT_SECONDS, gap_seconds=STA_GAP_SECONDS):
    cutoff = action_start - gap_seconds
    if cutoff <= 0:
        return None
    return max(0.0, cutoff - context_seconds), cutoff


def sample_sta_frames(video_path, action_start, frame_count=STA_FRAME_COUNT):
    window = sta_window(action_start)
    if window is None:
        return np.empty((0,), dtype=np.float32), np.empty((0,), dtype=np.int64)

    window_start, cutoff = window
    requested_times = np.linspace(window_start, cutoff, frame_count, endpoint=False)
    frames = []
    for requested_time in requested_times:
        frame = read_frame(video_path, float(requested_time))
        if frame.shape[1] != FRAME_WIDTH or frame.shape[0] != FRAME_HEIGHT:
            frame = cv2.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT), interpolation=cv2.INTER_AREA)
        frames.append(frame.astype(np.float32) / 255.0)
    frames = np.stack(frames)
    if frames.shape[1:] != (FRAME_HEIGHT, FRAME_WIDTH, 3):
        raise RuntimeError(f"Unexpected sampled frame shape: {frames.shape[1:]}")
    if frames.min() < 0.0 or frames.max() > 1.0:
        raise RuntimeError("Sampled RGB values are outside the expected [0, 1] range.")
    return frames, requested_times


video_by_id = index_videos(RAW_DIR)
train_with_video = match_available_annotations(train_df, video_by_id)
train_with_video["start_seconds"] = train_with_video["start_timestamp"].map(timestamp_to_seconds)
eligible = train_with_video[train_with_video["start_seconds"] >= STA_CONTEXT_SECONDS + STA_GAP_SECONDS]
if eligible.empty:
    raise RuntimeError("No local training video has a full five-second STA context; check the data paths.")

sample = eligible.iloc[0]
video_path = video_by_id[str(sample["video_id"]).upper()]
sta_frames, decoded_times = sample_sta_frames(video_path, sample["start_seconds"])
sta_start, sta_cutoff = sta_window(sample["start_seconds"])
assert sta_frames.shape[0] == STA_FRAME_COUNT
assert np.all(decoded_times < sta_cutoff)
assert sample_sta_frames(video_path, 0.76)[0].size == 0

# One-example batch in N,T,H,W,C order; labels remain annotation class IDs.
sta_batch = sta_frames[np.newaxis, ...]
verb_targets = np.asarray([sample["verb_class"]], dtype=np.int64)
noun_targets = np.asarray([sample["noun_class"]], dtype=np.int64)

print(f"Event: {sample['narration_id']} ({sample['video_id']})")
print(f"Observation window: {sta_start:.3f}s to {sta_cutoff:.3f}s")
print(f"Action starts: {sample['start_seconds']:.3f}s; required gap: {sample['start_seconds'] - sta_cutoff:.3f}s")
print(f"Batch shape: {sta_batch.shape}; dtype: {sta_batch.dtype}; pixel range: [{sta_batch.min():.3f}, {sta_batch.max():.3f}]")
print(f"Verb target: {verb_targets[0]}; noun target: {noun_targets[0]}")
print(f"Requested frame times: {np.round(decoded_times, 3)}")

plt.figure(figsize=(6, 4))
plt.imshow(sta_frames[0])
plt.title(f"STA context for {sample['narration_id']} (first sampled frame)")
plt.axis("off");

## 3. Evaluate a frequency-prior baseline

The baseline returns training-set verb and noun frequencies as score vectors for every validation event. Joint action scores below use the product of the verb and noun scores over all `97 x 300` pairs. Confirm this matches the course evaluator before treating the action metric as directly comparable to competition results.

In [ ]:
def normalized_prior(labels, class_count):
    labels = labels.to_numpy(dtype=np.int64)
    if np.any((labels < 0) | (labels >= class_count)):
        raise ValueError(f"Class ID outside expected range 0..{class_count - 1}.")
    counts = np.bincount(labels, minlength=class_count).astype(np.float64)
    return counts / counts.sum()


def top_k_recall(labels, scores, k):
    labels = np.asarray(labels, dtype=np.int64)
    top_k = np.argpartition(scores, -k, axis=1)[:, -k:]
    hits = np.any(top_k == labels[:, None], axis=1)
    return float(hits.mean())


def mean_present_class_top_k_recall(labels, scores, k):
    labels = np.asarray(labels, dtype=np.int64)
    top_k = np.argpartition(scores, -k, axis=1)[:, -k:]
    class_recalls = [
        np.any(top_k[labels == class_id] == class_id, axis=1).mean()
        for class_id in np.unique(labels)
    ]
    return float(np.mean(class_recalls))


verb_prior = normalized_prior(train_df["verb_class"], NUM_VERBS)
noun_prior = normalized_prior(train_df["noun_class"], NUM_NOUNS)
validation_verbs = validation_df["verb_class"].to_numpy(dtype=np.int64)
validation_nouns = validation_df["noun_class"].to_numpy(dtype=np.int64)
verb_scores = np.broadcast_to(verb_prior, (len(validation_df), NUM_VERBS))
noun_scores = np.broadcast_to(noun_prior, (len(validation_df), NUM_NOUNS))

joint_action_scores = (verb_prior[:, None] * noun_prior[None, :]).reshape(-1)
top_action_ids = np.argsort(joint_action_scores)[-5:][::-1]
validation_action_ids = validation_verbs * NUM_NOUNS + validation_nouns
action_top1 = float(np.mean(validation_action_ids == top_action_ids[0]))
action_top5_hits = np.isin(validation_action_ids, top_action_ids)
action_class_recalls = [
    action_top5_hits[validation_action_ids == action_id].mean()
    for action_id in np.unique(validation_action_ids)
]
action_mt5r = float(np.mean(action_class_recalls))

metrics = pd.DataFrame(
    {
        "Top-1 accuracy": [
            top_k_recall(validation_verbs, verb_scores, 1),
            top_k_recall(validation_nouns, noun_scores, 1),
            action_top1,
        ],
        "Top-5 / MT5R": [
            mean_present_class_top_k_recall(validation_verbs, verb_scores, 5),
            mean_present_class_top_k_recall(validation_nouns, noun_scores, 5),
            action_mt5r,
        ],
    },
    index=["Verb", "Noun", "Action"],
)
display(metrics)
print(f"Primary STA metric, Action MT5R: {action_mt5r:.4f}")
print(f"Validation action classes represented: {len(action_class_recalls)}")
print("Top-5 prior action pairs (verb_class, noun_class):", [(int(i // NUM_NOUNS), int(i % NUM_NOUNS)) for i in top_action_ids])

## Smoke-test checklist

- The frame batch has shape `(1, 16, 360, 640, 3)`, uses `float32` values in `[0, 1]`, and its labels match the selected training annotation.
- All decoded frame times are at or before `action_start - 1`; the early-action check returns no frames.
- The prior baseline produces verb, noun, and action metrics on validation labels without reading validation video.

Once these checks pass on the local sample and again on VSC, the next step is to replace the prior with a video model while keeping this input window, split, and evaluator fixed.